# Ising Probabilistic Computing on PYNQ

두 단계 운용 흐름:

1. **Annealing** — beta: 0 → beta_final (선형 증가)
   - 매 sweep 결과를 `sample_buffer` 에 기록

2. **Measurement** — beta = beta_final 고정
   - 매 sweep 결과를 `histogram` 에 누적

이 노트북은 먼저 SW 에뮬레이션으로 알고리즘을 검증하고,
PYNQ 보드에서 HW 가속 실행하는 방법을 보여준다.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'pynq_prob_computing'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from python.emulation import IsingEmulator, N_SPINS, HIST_BINS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print(f"N_SPINS = {N_SPINS}  HIST_BINS = {HIST_BINS}")

## 1. 모델 정의 — Ferromagnet (검증용)

`J_{ij} = +1` (i≠j), `h = 0`.
기저 상태: 모든 스핀 정렬 → `0x00` (모두 -1) 또는 `0xFF` (모두 +1).
높은 beta 에서 이 두 상태가 histogram 을 지배해야 한다.

In [ ]:
emu = IsingEmulator.ferromagnet(J_val=1, seed=42)
print("J =")
print(emu.J)
print("h =", emu.h)

## 2. 파라미터 설정

In [ ]:
# ── 두 단계 레지스터 파라미터 ──────────────────────────────────
N_ANNEAL_SWEEPS = 2_000   # 어닐링 총 sweep 수 → sample_buffer 크기
N_MEAS_SWEEPS   = 5_000   # 측정 총 sweep 수 → histogram 누적 수
BETA_FINAL      = 3.0     # 최종 역온도 (높을수록 저온, 기저 상태 집중)
N_ANNEAL_STEPS  = 20      # beta 를 몇 단계로 나눠 증가할지

# beta 인코딩 확인 (고정소수점 검증)
raw = IsingEmulator.encode_beta(BETA_FINAL)
decoded = IsingEmulator.decode_beta(raw)
print(f"beta_final={BETA_FINAL}  encoded=0x{raw:04X}  decoded={decoded:.6f}")
print(f"고정소수점 오차: {abs(BETA_FINAL - decoded):.2e}")

## 3. SW 에뮬레이션 실행

In [ ]:
result = emu.run(
    n_anneal_sweeps=N_ANNEAL_SWEEPS,
    n_meas_sweeps=N_MEAS_SWEEPS,
    beta_final=BETA_FINAL,
    n_anneal_steps=N_ANNEAL_STEPS,
    verbose=True,
)
print(f"\nsample_buffer.shape : {result.sample_buffer.shape}")
print(f"histogram.sum()      : {result.histogram.sum()} (= N_MEAS_SWEEPS)")
print(f"most_probable_state  : 0b{result.most_probable_state():08b}  (0x{result.most_probable_state():02X})")
print(f"gs_fraction          : {result.ground_state_fraction():.4f}")

## 4. 시각화

### 4-a. Annealing: 시계열 + beta 스케줄

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)

# ── (1) Spin state time series ─────────────────────────────────
ax = axes[0]
ax.imshow(
    np.unpackbits(result.sample_buffer[:, None], axis=1, bitorder='little').T,
    aspect='auto', cmap='RdBu', interpolation='nearest',
    extent=[0, len(result.sample_buffer), -0.5, N_SPINS - 0.5],
    vmin=0, vmax=1,
)
ax.set_ylabel('Spin index')
ax.set_title('Annealing — Spin state per sweep (blue=−1, red=+1)')
ax.set_xlabel('Sweep')

# ── (2) Energy during annealing ────────────────────────────────
ax = axes[1]
energies = result.energy_history(emu.J, emu.h)
ax.plot(energies, lw=0.8, color='steelblue')
ax.axhline(y=-28, ls='--', color='red', lw=1, label='Ground state (E=−28)')
ax.set_ylabel('Energy')
ax.set_title('Annealing — Hamiltonian energy')
ax.set_xlabel('Sweep')
ax.legend()

# ── (3) Beta schedule ──────────────────────────────────────────
ax = axes[2]
steps = np.arange(1, N_ANNEAL_STEPS + 1)
ax.step(steps, result.beta_schedule, where='post', color='darkorange', lw=2)
ax.set_ylabel('beta')
ax.set_xlabel('Annealing step')
ax.set_title(f'Beta schedule: 0 → {BETA_FINAL} in {N_ANNEAL_STEPS} steps')
ax.set_xlim(1, N_ANNEAL_STEPS)

plt.tight_layout()
plt.savefig('annealing_result.png', dpi=120)
plt.show()

### 4-b. Measurement: Histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Full histogram ─────────────────────────────────────────────
ax = axes[0]
ax.bar(np.arange(HIST_BINS), result.histogram, width=1,
       color='steelblue', alpha=0.8)
ax.axvline(x=0x00, color='red',  ls='--', lw=1.5, label='GS: 0x00 (all −1)')
ax.axvline(x=0xFF, color='lime', ls='--', lw=1.5, label='GS: 0xFF (all +1)')
ax.set_xlabel('Spin state (8-bit integer)')
ax.set_ylabel('Count')
ax.set_title(f'Measurement histogram  (beta={BETA_FINAL}, {N_MEAS_SWEEPS} sweeps)')
ax.legend()

# ── Top-20 states ──────────────────────────────────────────────
ax = axes[1]
top_idx  = np.argsort(result.histogram)[::-1][:20]
top_cnt  = result.histogram[top_idx]
labels   = [f'0b{i:08b}' for i in top_idx]
bars = ax.bar(np.arange(len(top_idx)), top_cnt, color='steelblue', alpha=0.8)
ax.set_xticks(np.arange(len(top_idx)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_ylabel('Count')
ax.set_title('Top-20 most visited spin states')

plt.tight_layout()
plt.savefig('histogram_result.png', dpi=120)
plt.show()

print(f"Ground-state fraction: {result.ground_state_fraction():.4f}")
print(f"Expected for beta={BETA_FINAL} ferromagnet: >0.5")

## 5. Beta 의존성 (온도 스캔)

beta 를 바꿔가며 gs_fraction 을 측정한다.  
beta 가 높을수록 기저 상태에 집중되어야 한다.

In [ ]:
betas       = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
gs_fracs    = []

for bf in betas:
    r = emu.run(n_anneal_sweeps=1000, n_meas_sweeps=3000,
                beta_final=bf, n_anneal_steps=10, verbose=False)
    gs_fracs.append(r.ground_state_fraction())
    print(f"beta={bf:.1f}  gs_frac={gs_fracs[-1]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(betas, gs_fracs, 'o-', color='steelblue', lw=2)
plt.xlabel('beta_final')
plt.ylabel('Ground-state fraction')
plt.title('Ferromagnet: gs_fraction vs beta')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('beta_scan.png', dpi=120)
plt.show()

## 6. SK Spin Glass (어려운 조합 최적화)

Shannington-Kirkpatrick 모델: 무작위 coupling.
기저 상태 에너지가 음수인지 확인한다.

In [ ]:
sk  = IsingEmulator.random_sk(J_std=1.0, seed=7)
r_sk = sk.run(n_anneal_sweeps=3000, n_meas_sweeps=5000,
              beta_final=4.0, n_anneal_steps=30, verbose=True)

gs_state = r_sk.most_probable_state()
s = np.array([(gs_state >> i) & 1 for i in range(N_SPINS)], dtype=np.int8) * 2 - 1
E_gs = -0.5 * float(s @ sk.J @ s)
print(f"\nMost probable state: 0b{gs_state:08b}  E={E_gs:.1f}")

plt.figure(figsize=(8, 4))
plt.bar(np.arange(HIST_BINS), r_sk.histogram, width=1, alpha=0.8)
plt.xlabel('Spin state')
plt.ylabel('Count')
plt.title('SK Spin Glass Measurement Histogram')
plt.tight_layout()
plt.show()

## 7. PYNQ 하드웨어 실행

이 섹션은 **PYNQ 보드에서만** 동작한다.  
Vivado HLS 로 합성한 비트스트림(`ising_sa.bit`)이 필요하다.

```bash
# 보드에서 실행
# 1. 비트스트림 복사
scp ising_sa.bit ising_sa.hwh xilinx@pynq:/home/xilinx/
# 2. 노트북 열기 → 아래 셀 실행
```

In [ ]:
# PYNQ 보드에서만 실행 가능
try:
    from python.pynq_driver import IsingOverlay
    import numpy as np

    BITFILE = '/home/xilinx/ising_sa.bit'

    J = np.ones((8, 8), dtype=np.int32) - np.eye(8, dtype=np.int32)
    h = np.zeros(8, dtype=np.int32)

    with IsingOverlay(BITFILE) as hw:
        hw.configure(
            J=J, h=h,
            beta_final=3.0,
            n_anneal_sweeps=10_000,
            n_meas_sweeps=50_000,
            n_anneal_steps=20,
            lfsr_seed=0xABCD1234,
        )
        result_hw = hw.run(timeout=120)

    print(f"HW gs_fraction : {result_hw.ground_state_fraction():.4f}")
    print(f"HW peak state  : 0b{result_hw.most_probable_state():08b}")

except Exception as e:
    print(f"[PYNQ not available] {e}")
    print("SW 에뮬레이션 결과를 사용하세요.")